# 03 — Run the same ten studies through both JEV models

**Purpose:** carry out a reproducible technical pilot using JEV-9B first and JEV-27B v1.1 second. Each model has its own explicit execution cell below. There is no need to change the model selection or rerun the first model's cell to run the second model.

| Protocol element | Specification |
|---|---|
| Source | Prepared JSONL selected in the two evaluation YAML profiles |
| Pilot sample | Ten studies sampled without replacement; seed 42; one shared frozen ordered sample |
| Model inputs | Original study ID, title and abstract; no human decisions or ITEM_ID |
| Eligibility criteria | The same version-controlled `retain_for_fulltext` question for both models |
| Execution | Hosted Inference.sh; sequential requests; one study and one question per request |
| Limits | At most ten attempts per model, twenty across both completed pilot runs; no automatic retries |
| Outputs | Separate model directories, raw responses, validated probabilities, coverage, errors and provenance |

Both model execution flags are **true**. Executing the model cells, including through Run All, may incur API charges. Loading settings, preparing data, reviewing plans and generating the methods summary do not make API calls. Preparing the sample can write a private manifest. This is a technical pilot; probabilities alone do not establish screening accuracy.

Run the numbered sections from top to bottom. Notebook 02's separate one-state test remains unchanged.

## 1. Load both model profiles

Use this project's `.venv` kernel and restart it after source-code changes. The two model selections are explicit in the next cell. Settings remain in YAML; reusable loading, inference, checkpointing and reporting logic remains in `src/jev_eval/`.

Both profiles use the same prepared dataset, sampling seed and eligibility file. They select different existing model YAMLs and output directories. Initialization loads `.env` without printing `INFERENCE_API_KEY`. This step sends no requests. Changing model app IDs does not pin an immutable weight revision; returned model identifiers are retained separately.

In [ ]:
try:
    from jev_eval.environment import initialize_notebook
except ModuleNotFoundError:
    raise RuntimeError(
        "Select this project's .venv Python kernel. From the repository terminal, "
        "run: uv sync --locked --group notebooks"
    ) from None

from jev_eval.config import PrerequisiteError
from jev_eval.data import prepare_pilot_sample
from jev_eval.reporting import (
    paired_pilot_plan_table,
    paired_pilot_probability_table,
    pilot_methods_summary,
    pilot_run_status,
)
from jev_eval.runner import PilotPersistenceError, plan_run, run_experiment

# Both model selections are fixed here; no switching is needed later.
project_9b = initialize_notebook(workflow_path="configs/workflows/pilot_jev_9b.yaml")
project_27b = initialize_notebook(workflow_path="configs/workflows/pilot_jev_27b.yaml")
result_9b = None
result_27b = None

## 2. Freeze one ten-study sample

**File-writing warning:** `ALLOW_SAMPLE_WRITES = True` permits creation of `results/pilots/depression-10/sample.json`. It records the source checksum, ordered study IDs, seed, validation audit and eligibility fingerprint. Existing manifests are verified and never overwritten. This cell makes no API calls.

The current source contains 7,008 studies, including eleven without a title or abstract. Those eleven remain in the source and human reference data and are reported as unavailable for inference. Ten studies are sampled from the 6,997 with some citation text. Title-only records remain available. Other validation errors block sampling.

The single `sample` variable below supplies **both** model plans. Each plan also verifies its YAML-selected data and criteria against the same frozen manifest. Human screening labels are not used to select the sample or construct requests.

In [ ]:
ALLOW_SAMPLE_WRITES = True
pilot = prepare_pilot_sample(project_9b, allow_writes=ALLOW_SAMPLE_WRITES)
sample = pilot.states

display(pilot.summary())
print("Frozen sample manifest:", pilot.path)

## 3. Review both plans and the shared eligibility question

No requests are sent here. Confirm ten studies and at most ten attempts for **each** model. The sample and task hashes must match, and the app IDs and output directories must differ. The package checks these conditions before displaying the table.

A NOUL is a yes/no question answered with a probability. The exact shared instructions and true/false criteria are displayed below for review and methods reporting. No eligibility cutoff is applied to the returned probabilities.

**Cost check:** twenty attempts is a request ceiling, not a monetary budget. Check provider prices, account limits and other account activity before execution. The runner spaces starts by at least 3.1 seconds within each model run. Input-token usage and HTTP elapsed time do not establish billed cost. The YAML's generic generation fields are not sent to Inference.sh.

In [ ]:
import pandas as pd

# The same ordered sample and shared task are used in both plans.
plan_9b = plan_run(project_9b, sample)
plan_27b = plan_run(project_27b, sample)
display(paired_pilot_plan_table(plan_9b, plan_27b))

display(
    pd.DataFrame(
        [
            {"question_id": name, **question.model_dump()}
            for name, question in plan_9b.task.questions.items()
        ]
    )
)

## 4. Run JEV-9B on the ten studies — potentially billable

**Paid API warning:** setting `RUN_JEV_9B = True` and executing the next cell submits up to ten requests to `autotrust/jev-9b`. It may incur charges. The YAML paid-call permission must also be enabled. This cell runs only JEV-9B.

**File-writing warning:** the call requires and enables result saving under `results/pilots/depression-10/jev-9b/`. Requests, exact response bytes, normalized probabilities and provenance are saved incrementally. No automatic retry occurs.

The first failed or ambiguous request stops this model's run. Unknown outcomes preserve pending state and block resume. If saving fails, keep the kernel open: the exception's retained result is kept as `result_9b`. Inspect saved receipts and pending state rather than submitting again with a new run ID.

In [ ]:
RUN_JEV_9B = True  # Enabled: executing this cell may incur API charges.

if RUN_JEV_9B:
    try:
        result_9b = run_experiment(
            plan_9b,
            allow_paid_calls=RUN_JEV_9B,
            allow_writes=True,
        )
    except PilotPersistenceError as error:
        result_9b = error.result
        print(str(error))
    except PrerequisiteError as error:
        print("JEV-9B could not start or resume:", error)
else:
    print("JEV-9B skipped: set RUN_JEV_9B = True to submit the reviewed pilot.")

## 5. Review JEV-9B before proceeding

This is a read-only status summary. Inspect successful, failed, unknown and unrecorded studies, error codes, pending state, reported input-token usage and the saved directory. Review provider charges before enabling JEV-27B. Resolve unknown outcomes before continuing.

`Not executed in this kernel` means no result object has been supplied here; it does not assert that no previous saved run exists. Partial runs remain visible and are not described as complete.

In [ ]:
display(pilot_run_status(plan_9b, result_9b))

## 6. Run JEV-27B v1.1 on the same ten studies — potentially billable

**Paid API warning:** setting `RUN_JEV_27B = True` and executing the next cell submits up to ten requests to `autotrust/jev-27b-v1-1` and may incur charges. Review the JEV-9B run first. This cell runs only JEV-27B; it does not invoke the JEV-9B execution cell.

The pre-built `plan_27b` uses the identical ordered sample and eligibility question checked in Section 3. There is no new sampling step or model-selection edit.

**File-writing warning:** responses and provenance are saved incrementally under the separate `results/pilots/depression-10/jev-27b-v1-1/` directory. The same error, pending-state and persistence safeguards apply. If saving fails, keep `result_27b` and the kernel available for investigation.

In [ ]:
RUN_JEV_27B = True  # Enabled: executing this cell may incur API charges.

if RUN_JEV_27B:
    try:
        result_27b = run_experiment(
            plan_27b,
            allow_paid_calls=RUN_JEV_27B,
            allow_writes=True,
        )
    except PilotPersistenceError as error:
        result_27b = error.result
        print(str(error))
    except PrerequisiteError as error:
        print("JEV-27B v1.1 could not start or resume:", error)
else:
    print("JEV-27B v1.1 skipped: set RUN_JEV_27B = True when ready.")

## 7. Review JEV-27B v1.1

This read-only summary uses only the second model's result. A failure, unknown outcome or missing response remains explicit. Stored input-token usage is what the provider reported; it is not a monetary cost estimate.

In [ ]:
display(pilot_run_status(plan_27b, result_27b))

## 8. Inspect paired probabilities and coverage

The next cell joins results by original study ID in the frozen sample order, giving one row per study and separate probability/status columns for both models. It does not assume that response row positions match. Failed, unknown or unrecorded responses have no probability; they are never filled with zero or treated as exclusions.

This table is useful for checking outputs and model agreement. It does not calculate accuracy or assign inclusion decisions. Binary evaluation needs a separately specified threshold and linkage to human reference labels.

**Privacy warning:** the table contains real study IDs and model outputs. Keep executed notebooks private and clear all outputs before committing. Citation text, API credentials and raw HTTP bodies are not displayed by this table.

In [ ]:
paired_probabilities = paired_pilot_probability_table(plan_9b, result_9b, plan_27b, result_27b)
display(paired_probabilities)

## 9. Generate a methods description and provenance table

The package generates a draft from the frozen sample, validated plans and available results. It reports sampling, input fields, eligibility criteria, models, request limits, pacing, saved artifacts and recovery rules. Actual counts and identifiers come from the supplied result objects.

Before both ten-study runs complete successfully, the draft is explicitly labelled as a **planned protocol with incomplete execution**. Only two complete runs produce a completed-pilot methods description. Review the wording for your paper and report failures and missingness where present. Reproducible sampling and recorded settings do not guarantee identical outputs from a mutable hosted service.

The accompanying table provides source/sample/task checksums, seed, requested run IDs, lockfile identity, Git revision, returned model names and run timestamps. An uncommitted code state is reported honestly. Full instructions are shown in Section 3; exact responses and further provenance are in the private run artifacts. No files or API requests are created by this reporting cell.

In [ ]:
methods = pilot_methods_summary(pilot, plan_9b, plan_27b, result_9b, result_27b)
print(methods.text)
display(methods.provenance)

## 10. Optional raw-response inspection — may display private data

Raw-response display remains disabled by default in both workflows. Enable it only when needed and clear outputs before Git. Responses may contain private study text. This cell reads retained results and makes no API requests.

In [ ]:
if result_9b is not None and project_9b.workflow.privacy.display_raw_responses:
    display(result_9b.responses[:1])
else:
    print("JEV-9B raw-response display is disabled or no result is available.")

if result_27b is not None and project_27b.workflow.privacy.display_raw_responses:
    display(result_27b.responses[:1])
else:
    print("JEV-27B raw-response display is disabled or no result is available.")

## 11. Resume and preserve the research record

Use the same model workflow and run ID when resuming, with the same source, sample, task, code and locked environment. Run Sections 1–3 to rebuild both plans, then enable only the model execution cell you intend to resume. Saved successes and known terminal failures are skipped, preventing routine duplicate submissions. Changed scientific-input or source fingerprints block resume.

Unknown outcomes, unfinished tasks and uncertain server responses preserve pending state and block automatic resume. Inspect task IDs, receipts and the provider dashboard; do not delete a pending marker or choose a fresh run ID to force another POST. A killed process can leave an exclusive `.<run-id>.lock` beside its run directory. Investigate the process and pending state before manually removing a stale lock. These local safeguards are conservative recovery, not a server-side exactly-once guarantee.

Each model directory retains `manifest.json`, `predictions.jsonl`, `responses.jsonl`, `failures.jsonl`, and `raw/` receipt JSON plus exact HTTP `.body` files. The shared sample manifest links both runs. All are ignored by Git. Notebook 04's Boolean metrics require a separately agreed threshold/reference-label step for these probability outputs. No API keys, real data or executed outputs belong in source control.

## 12. Full dataset inference — overview

**Start the full workflow here from a fresh kernel.** The earlier 22 cells are preserved. Their pilot execution flags are currently enabled, so do not use **Run All** to start this evaluation. Run only Sections 12–20. The new full-run flags below are independent and initially false.

The objective is to collect JEV-9B (`autotrust/jev-9b`) and JEV-27B v1.1 (`autotrust/jev-27b-v1-1`) probabilities for the complete prepared human-screened cohort through the existing Inference.sh backend. The JSONL contains only original study ID, title and abstract. Human `Inclusion Status` and `ITEM_ID` remain in a separate reference CSV from the preparation bundle and must never enter model inputs.

Both models use the identical ordered states and existing version-controlled eligibility question. No new criteria or decision threshold are introduced. Every source study is retained in the cohort. A record without any title or abstract gets an explicit `not_inferred_no_text` result and null probability, without a paid request or a changed human decision. Other validation errors block the plan.

The shared serial runner saves each received response and outcome incrementally in separate full-evaluation model directories. Unknown outcomes block automatic resume; recorded terminal results are not submitted again. This workflow is separate from the completed ten-study pilots and does not import their results automatically. No API calls are made by configuration loading, validation, planning or reporting.

In [ ]:
# Independent full-run objects: no earlier pilot cells need to be executed.
import pandas as pd

from jev_eval.config import PrerequisiteError
from jev_eval.data import load_full_dataset
from jev_eval.environment import initialize_notebook
from jev_eval.evaluation import evaluation_ready_table, export_evaluation_ready, load_reference_csv
from jev_eval.reporting import (
    full_run_metadata_table,
    full_run_plan_table,
    load_full_run_if_present,
    print_full_progress,
    validate_full_results,
)
from jev_eval.runner import PilotPersistenceError, plan_run, run_experiment

full_project_9b = initialize_notebook(workflow_path="configs/workflows/full_jev_9b.yaml")
full_project_27b = initialize_notebook(workflow_path="configs/workflows/full_jev_27b.yaml")
full_result_9b = None
full_result_27b = None

## 13. Load and validate the complete dataset

The full profiles load the JSONL exported by data preparation. The actual count is determined from the file; there is no expected-count constant and no sampling step. Required fields, string IDs, duplicate/missing IDs, text types and the input-field allowlist are checked. The allowlist rejects human decisions, ITEM_ID and unexpected input fields.

The summary reports missing titles/abstracts and records with no text without silently removing studies. The byte checksum identifies the exact input file; the ordered-state hash identifies normalized content and order. If a conversion manifest is available, its states checksum is verified too.

The reference CSV is loaded separately to validate linkage and label encoding before spending on inference. Its IDs stay strings, including leading zeros. Only the configured 0/1 mapping is accepted; missing labels remain missing. Shared or missing ITEM_ID values are reported. The join key will be `study_id`, not ITEM_ID. These cells neither write files nor make API calls.

In [ ]:
full_dataset = load_full_dataset(full_project_9b)
full_states = full_dataset.states
full_references = load_reference_csv(full_project_9b, full_states)
display(full_dataset.summary())
display(full_references.summary())
print("Complete dataset records:", len(full_states))
print("Dataset filename:", full_dataset.manifest["source_file"])
print("Dataset SHA-256:", full_dataset.manifest["source_file_sha256"])

## 14. Confirm screening instructions and experiment configuration

Both plans below receive the same complete `full_states` list. Each independently verifies it against its YAML-selected source and task. The exact existing NOUL instructions and true/false criteria are shown for methodological review. A NOUL returns P(true), not a generated text answer.

The configuration table lists file paths/checksums. Requested apps come from the existing model YAMLs; output names reported by the provider may differ in capitalization or naming. Both are retained. Temperature, generation-token limits and a model seed are not sent to this decision endpoint. The configured decision threshold is null, so this stage preserves probabilities rather than assigning inclusion decisions.

In [ ]:
full_plan_9b = plan_run(full_project_9b, full_states)
full_plan_27b = plan_run(full_project_27b, full_states)

display(
    pd.DataFrame(
        [
            {"question_id": name, **question.model_dump()}
            for name, question in full_plan_9b.task.questions.items()
        ]
    )
)
display(
    pd.DataFrame(
        [
            {"model": plan.project.models[0].name, "configuration_file": path, "sha256": digest}
            for plan in (full_plan_9b, full_plan_27b)
            for path, digest in plan.configuration_sha256.items()
        ]
    )
)

## 15. Full-run configuration, dry run and safety checks

**Dry run is the initial mode.** The next cell checks both plans and reads saved progress, without contacting Inference.sh or creating result files. Inspect actual record/API counts, pending studies, unknown outcomes, fingerprint compatibility, attempt budgets and model-specific output directories. Request counts are not monetary cost estimates.

Defaults are sequential execution, at least 3.1 seconds between starts, a 120-second request timeout, at most three attempts per study/model, and explicit per-model request/attempt ceilings. Only explicit 429 rejections are retried, with exponential backoff and the provider's Retry-After delay when present. Unknown outcomes are never automatically reposted. Individual terminal errors remain recorded; authentication/billing/throttling exhaustion or repeated consecutive failures pause the run. Provider account activity can impose additional limits.

Before live execution, deliberately edit the **dedicated full workflow YAMLs**: set `inference_sh_full.dry_run: false`, `execution.allow_paid_calls: true`, and `output.allow_writes: true` for the model you intend to run. Then rerun Sections 12–15 to reload/review the settings. Nothing in this notebook changes these permissions automatically or proceeds from planning into live inference.

**File-writing warning for live execution:** the runner creates or verifies the shared cohort manifest and writes results under `results/full/depression/full-depression-v1/`, with separate model subdirectories. Keep the same reviewed experiment/run IDs when resuming; choose a new explicitly reviewed experiment directory only for a different scientific run. Concurrent writers are blocked by a lock. A source/task/code mismatch or pending unknown request requires investigation, not a new ID to bypass recovery.

In [ ]:
full_execution_plan = full_run_plan_table(full_plan_9b, full_plan_27b)
display(full_execution_plan)
print(
    "Baseline API submissions across both models:", int(full_execution_plan.logical_requests.sum())
)
print(
    "Maximum total attempt budgets:",
    sum(
        project.workflow.execution.max_total_attempts
        for project in (full_project_9b, full_project_27b)
    ),
)
print("Dry-run planning complete. No API call was made and no live flag was enabled.")

## 16. Run JEV-9B on the complete dataset — potentially billable

**Paid API warning:** setting `RUN_FULL_JEV_9B = True` and executing this cell may submit every pending eligible study to Inference.sh and incur charges. YAML paid permission must also be enabled and dry_run disabled. This cell never runs JEV-27B.

**File-writing warning:** persistence is required and enabled in the call, with separate full-run paths. Progress shows counts only. Raw HTTP bodies, compact result rows, task IDs, attempts and errors are saved incrementally. Textless records are retained without an API request. If storage fails, keep the kernel open: `full_result_9b` retains the in-memory run and latest exchange. Investigate saved pending state before resuming; never automatically repeat an uncertain POST.

In [ ]:
RUN_FULL_JEV_9B = True

if RUN_FULL_JEV_9B:
    try:
        full_result_9b = run_experiment(
            full_plan_9b,
            allow_paid_calls=RUN_FULL_JEV_9B,
            allow_writes=True,
            progress=print_full_progress,
        )
    except PilotPersistenceError as error:
        full_result_9b = error.result
        print(str(error))
    except PrerequisiteError as error:
        print("Full JEV-9B execution blocked:", error)
else:
    print("Full JEV-9B execution is disabled. Review Section 15 before enabling it.")

## 17. Run JEV-27B v1.1 on the same complete dataset — potentially billable

Review the JEV-9B outcome, provider charges and validation before continuing. This plan uses exactly the same ordered cohort and eligibility question, changing the model-specific selection and output directory.

**Paid API warning:** setting `RUN_FULL_JEV_27B = True` and executing this cell submits pending eligible studies to `autotrust/jev-27b-v1-1` and may incur charges. Its own YAML permissions/dry-run setting must allow execution. The cell does not invoke JEV-9B.

**File-writing warning:** separate results are saved incrementally under the JEV-27B full-run directory. The same retry budgets, pending markers, source checks, progress reporting and recovery rules apply. Preserve the kernel's `full_result_27b` after any storage failure.

In [ ]:
RUN_FULL_JEV_27B = True

if RUN_FULL_JEV_27B:
    try:
        full_result_27b = run_experiment(
            full_plan_27b,
            allow_paid_calls=RUN_FULL_JEV_27B,
            allow_writes=True,
            progress=print_full_progress,
        )
    except PilotPersistenceError as error:
        full_result_27b = error.result
        print(str(error))
    except PrerequisiteError as error:
        print("Full JEV-27B execution blocked:", error)
else:
    print("Full JEV-27B execution is disabled. Review Section 15 before enabling it.")

## 18. Validate saved inference outputs

This section reloads saved files **without resuming or sending requests**. It checks expected and recorded counts, successes, failed/unknown/textless/missing records, duplicate or unexpected IDs, status/probability validity, missing task/model metadata and incompatible provenance. Probabilities must be finite numeric values in 0–1.

Requested/returned name differences are reported separately from unexpected identities: known provider naming/capitalization variants are retained, never silently rewritten. Missing immutable model revisions remain null and are reported as unavailable. Errors and missing predictions stay in the summaries; they are not treated as negative screening decisions.

If a checkpoint is corrupt or its pending marker is unresolved, inspect private receipts and the provider dashboard before recovery. Read-only loading intentionally does not relax live resume guards. The saved objects use separate names so they do not replace an in-memory exchange retained after a storage failure.

In [ ]:
full_saved_9b = load_full_run_if_present(full_plan_9b)
full_saved_27b = load_full_run_if_present(full_plan_27b)
full_output_validation = pd.concat(
    [
        validate_full_results(full_plan_9b, full_saved_9b),
        validate_full_results(full_plan_27b, full_saved_27b),
    ],
    ignore_index=True,
)
display(full_output_validation)

## 19. Record and inspect reproducibility metadata

Each model manifest records experiment/run identity, requested app, exact shared cohort and question, input/configuration/source/lockfile checksums, resolved settings, Git state, environment versions, UTC timestamps and status. Execution sessions record settings used on explicit resumes. Per-study rows and receipts retain task IDs, available token usage, attempts, errors and exact response hashes/paths. Large response bodies stay on disk rather than accumulating in memory.

The table below uses saved run metadata when available and prospective plan metadata otherwise. Returned model names and any explicitly provider-reported revision are recorded; an immutable weight revision is **not verified** by a name or app version. Unreported revisions remain null. No API credentials or request authorization headers are exported. Keep raw artifacts and executed notebook outputs private. Hosted reproducibility means traceable inputs/settings/results, not guaranteed identical future responses.

In [ ]:
display(full_run_metadata_table((full_plan_9b, full_plan_27b), (full_saved_9b, full_saved_27b)))

## 20. Prepare an evaluation-ready dataset for notebook 04

This cell joins saved model outputs to the separate human CSV by original `study_id`, preserving ID strings and any ITEM_ID values. It creates one row per expected study/model, including missing, failed, unknown and textless outcomes with null probabilities. The reference join is validated as many-to-one; duplicate reference IDs, duplicate study/model results, unexpected identities or malformed successful predictions block export instead of being silently removed.

Human labels are mapped only according to the explicit versioned 0/1 mapping. Missing labels and ITEM_ID values remain missing. A shared ITEM_ID does not merge distinct studies. No threshold or performance metric is calculated here; statistical evaluation belongs in `04_evaluation.ipynb`.

The displayed table is aggregate only. The row-level `full_evaluation_ready` object contains private IDs and human decisions; do not display or commit it casually.

**File-writing warning:** `EXPORT_FULL_RESULTS = True` plus YAML write permission exports a uniquely named private bundle under the full experiment's `exports/` directory. It contains CSV, checksums and a validation/completeness manifest. Existing bundles are not overwritten. Review Section 18 first; partial coverage remains explicitly labelled in the export manifest. This cell makes no API calls.

In [ ]:
full_evaluation_ready = evaluation_ready_table(
    full_plan_9b, full_saved_9b, full_plan_27b, full_saved_27b, full_references
)
display(
    pd.DataFrame(
        [
            {
                "expected_study_model_rows": len(full_states) * 2,
                "actual_rows": len(full_evaluation_ready),
                "missing_human_label_rows": int(full_evaluation_ready.human_label.isna().sum()),
                "missing_probability_rows": int(
                    full_evaluation_ready.probability_true.isna().sum()
                ),
                "missing_ITEM_ID_rows": int(full_evaluation_ready.ITEM_ID.isna().sum()),
            }
        ]
    )
)

EXPORT_FULL_RESULTS = False
if EXPORT_FULL_RESULTS:
    full_export_directory = export_evaluation_ready(
        full_plan_9b,
        full_saved_9b,
        full_plan_27b,
        full_saved_27b,
        full_references,
        allow_writes=EXPORT_FULL_RESULTS,
    )
    print("Evaluation-ready export:", full_export_directory)
else:
    print("Export disabled. No files written by this cell.")